# 第9回　正則化
***
> **前提**: 第3回・第7回の回帰分析を発展させ，正則化とロジスティック回帰を学びます。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（機械学習）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. データの読み込み
2. Ridge / Lasso 回帰
3. ロジスティック回帰の正則化
4. 正則化強度の比較

---

## この回で学ぶこと

### 過学習（Overfitting）とは何か

モデルが訓練データに**過度に適合**してしまい，未知のデータ（テストデータ）では精度が落ちる現象を「過学習」という。特に特徴量が多い場合（変数の数 > サンプル数），線形モデルでも簡単に過学習が起きる。

```
【過学習の兆候】
訓練 R² = 0.98  テスト R² = 0.42  ← 大きく乖離 → 過学習

【理想的な状態】
訓練 R² = 0.82  テスト R² = 0.79  ← ほぼ同じ → 汎化できている
```

### 正則化の直感的な理解

正則化とは「モデルの係数が大きくなりすぎることにペナルティを課す」仕組みだ。

通常の線形回帰の損失関数：**誤差の二乗和（RSS）**を最小化

正則化では損失関数に「係数の大きさ」を加える：
- **Ridge（L2正則化）**: RSS + α × Σ(係数²)
- **Lasso（L1正則化）**: RSS + α × Σ|係数|

αが大きいほど正則化が強く，係数は0に近づく（モデルが単純になる）。

### Ridge vs Lasso の本質的な違い

| 比較軸 | Ridge（L2） | Lasso（L1） |
|---|---|---|
| 係数の扱い | すべての係数を小さくする（0にはしない） | 不要な特徴量の係数を**完全に0にする** |
| 特徴量選択 | できない | **できる**（スパースモデル） |
| 向いている場面 | 全特徴量が多少は予測に寄与する場合 | 重要な特徴量が少数に絞られる場合 |
| 計算 | 解析解あり（安定） | 反復計算が必要 |

> **卒業研究での活用**: 説明変数が多い（遺伝子発現データ，アンケート，センサーデータなど）場合，Lasso で重要変数を自動選択する手法は非常によく使われる。

### ロジスティック回帰の正則化パラメータ C

scikit-learn の `LogisticRegression` では正則化強度を **C** で指定する。Cは `alpha` の逆数だ：
- **C が小さい** → 正則化が強い → 係数が小さい → シンプルなモデル
- **C が大きい** → 正則化が弱い → 係数が自由 → 複雑なモデル（過学習しやすい）

混乱しやすいので注意：Ridge の `alpha` と `LogisticRegression` の `C` は**逆の関係**だ。

---

## 📚 将来の自分のためのメモ — 大学研究での応用


### どんな研究で使われるか

| 分野 | 具体例 |
|---|---|
| 生物学・医学 | **遺伝子発現データ**（変数が数万・サンプルが数十）→ Lasso で疾患関連遺伝子を選択 |
| 心理学・行動科学 | 多数のアンケート項目から，うつ傾向や学習成果に効く因子を絞り込む |
| 経済学・マーケティング | 価格弾力性，購買予測で変数が多い回帰（Ridge で安定化） |
| 医学診断 | ロジスティック回帰 + L1/L2 正則化でリスク因子のスクリーニング |

### 応用できる場面

- **説明変数がサンプル数より多い**（または多項式展開で膨らんだ）ときの過学習防止
- **解釈可能性**が重要（「どの変数が効いているか」を係数で読みたい）
- 線形関係が**おおまかに**成り立つ連続値・二値予測
- Lasso で**変数選択**し，その後に生物学的検証や追加実験を計画する流れ

### 応用しにくい・向かない場面

- **強い非線形・複雑な相互作用**（画像，音声，自然言語）→ 深層学習や木系の方が適する
- **因果効果の推定**が目的のとき（正則化回帰は「予測」向き。因果は傾向スコア，DAG など別手法）
- 係数の符号だけで「A が B の原因」と結論づけること（相関≠因果）
- サンプルが極端に少なく，かつノイズが大きい場合（正則化だけでは不十分なことも多い）

### 研究で報告するときのポイント

1. **正則化強度（α や C）の選び方**を交差検証で行ったことを書く（第13回と連動）
2. Lasso を使った場合は**選択された変数のリスト**と生物学的/理論的な解釈をセットで示す
3. 訓練 R² とテスト R² の**乖離**を報告し，過学習が抑えられていることを示す

### 関連する発展トピック（調べてみると良い）

- Elastic Net（Ridge + Lasso の中間）
- Group Lasso（遺伝子セット単位の選択）
- ベイズ回帰（不確かさ付きの係数推定）

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

DATA_BASE = "https://raw.githubusercontent.com/ShotaYmzk/AI-kadai/main/data"

def load_student_csv(filename: str) -> pd.DataFrame:
    return pd.read_csv(f"{DATA_BASE}/student/{filename}", sep=";")


## 問題1　ベースラインの線形回帰と過学習の観察　【骨格+選択】
***

### データについて

UCI の学生成績データには，ポルトガルの数学と国語の成績が含まれている。
- `G1`: 1学期の成績（0〜20点）
- `G2`: 2学期の成績（0〜20点）
- `G3`: 最終成績（0〜20点）← 予測したい目的変数

今回は `G3` を目的変数とし、`G1`・`G2` をはじめとする**多数の数値特徴量**を説明変数に使う。さらに `PolynomialFeatures` で特徴量を多項式展開して数を増やし、「特徴量が多いほど過学習しやすい」状況をわざと作って、正則化の効果を観察する。

### なぜ標準化が必要か（回帰の場合）

線形回帰自体はスケールに影響されないが，正則化（Ridge/Lasso）は**係数の大きさにペナルティ**をかけるため，スケールが異なると不公平なペナルティになる。問題2以降で Ridge/Lasso を使うため，ここで標準化しておく。

### 評価指標の意味

- **MSE（Mean Squared Error）**: 予測誤差の二乗の平均。値が小さいほど良い。単位は（得点²）なので直感的に分かりにくい
- **RMSE（Root MSE）**: MSEの平方根。元の単位（点）で誤差を表せる
- **R²（決定係数）**: 0〜1の範囲で，1が完璧な予測。「目的変数の分散のうちモデルが説明できる割合」

### 課題

下のコードセルは前処理（多項式展開・標準化）まで用意してありますが、**核心（線形回帰モデルの学習）はあなたが書きます**（`# ★あなたが書く★`）。書いて実行し、出力される「訓練 R²」と「テスト R²」、そして「係数の絶対値の最大」を観察してください。`PolynomialFeatures` で**わざと過学習しやすい状況**を作っています。

`DEGREE` を `1 → 2 → 3` と上げると、特徴量の数が増え、訓練 R² とテスト R² の差（＝過学習）がどう広がるかを確認できます（最低2通り試すとわかりやすいです）。

観察したうえで、次の **設計判断** に答えてください。

> **設計判断1**: 訓練 R² は高いのにテスト R² が低い（＝過学習している）状況です。この問題への対処として、次のどれが適切か**1つ選び、理由**を解答用コードセルに書いてください。ここでの選択は問題2の実験につながります。
>
> - **(A) 何もしない（正則化なしの `LinearRegression` のまま）**
> - **(B) Ridge（L2正則化）を使う** … すべての係数を小さく抑える
> - **(C) Lasso（L1正則化）を使う** … 不要な係数を0にして特徴量を絞る
> - **(D) ElasticNet（L1+L2 の両方）を使う** … Ridge と Lasso の中間
> - **(E) 特徴量を減らす（多項式の次数 `DEGREE` を下げる）** … モデル自体を単純にする
> - **(F) 学習データを増やす** … サンプル数を増やして暗記を防ぐ
>
> ヒント：過学習は「係数が極端に大きくなり、訓練データの細部まで暗記してしまう」ことで起きます。出力された「係数の絶対値の最大」と、L2/L1 が係数に何をするか（この回で学ぶこと参照）を根拠にしてください。「正解は1つ」ではなく、**根拠が筋の通った選択**であることが重要です。

In [ ]:
# === 完成済みコード：そのまま実行して、訓練 R² と テスト R² の差（過学習）を観察してください ===
df = load_student_csv("student-mat.csv")

# 数値特徴量を多めに使う（特徴量が多いほど過学習しやすい）
feature_cols = ["G1", "G2", "studytime", "failures", "absences",
                "age", "Medu", "Fedu", "traveltime", "freetime",
                "goout", "Dalc", "Walc", "health"]
X = df[feature_cols].values
y = df["G3"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

# === ★ここを変えて実験する★：DEGREE を 1 → 2 → 3 と上げると過学習が強まる ===
DEGREE = 2

# 特徴量を多項式展開して「わざと過学習しやすい状況」を作る
poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# 正則化を公平にかけるため標準化しておく（問題2以降でも X_train_s / X_test_s を再利用）
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_poly)
X_test_s = scaler.transform(X_test_poly)
print(f"特徴量の数: {X_train_s.shape[1]}（DEGREE={DEGREE}）")

# ★あなたが書く★：線形回帰モデル lr を作り、訓練データ（X_train_s, y_train）で学習する（2行）
#   ヒント: LinearRegression() でモデルを作り、.fit(説明変数, 目的変数) で学習
lr = ___
print(f"訓練 R²  = {r2_score(y_train, lr.predict(X_train_s)):.4f}")
print(f"テスト R² = {r2_score(y_test, lr.predict(X_test_s)):.4f}")
print(f"係数の絶対値の最大 = {np.abs(lr.coef_).max():,.1f}")


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
import pandas as pd


# (1-a) 観察（`DEGREE` を変えたときの R²。3通り以上）
observation_1_a = pd.DataFrame([
    {'degree': 1, 'n_features': None, 'train_r2': None, 'test_r2': None},
    {'degree': 2, 'n_features': None, 'train_r2': None, 'test_r2': None},
    {'degree': 3, 'n_features': None, 'train_r2': None, 'test_r2': None},
])

# (1-b) 観察：DEGREE を上げると訓練 R² とテスト R² の差はどうなったか
observation_1_b = """
"""

# (1-c) 観察：係数の絶対値の最大はどうなったか
observation_1_c = """
"""

# (1-d) 設計判断1：過学習への対処として選んだ選択肢（A〜F）：(　)
# (A) 何もしない（正則化なしの LinearRegression のまま）
# (B) Ridge（L2正則化）を使う
# (C) Lasso（L1正則化）を使う
# (D) ElasticNet（L1+L2 の両方）を使う
# (E) 特徴量を減らす（多項式の次数 DEGREE を下げる）
# (F) 学習データを増やす
design1_choice = ""

# (1-e) その理由（係数の大きさ・L2/L1 が係数に何をするかに触れて）
answer_1_e = """
"""



## 問題2　正則化強度 alpha の実験（Ridge と Lasso）　【実験】
***

### alpha の意味と選び方

`alpha` は正則化の強さを決めるハイパーパラメータ（事前に設定する値）であり，通常は交差検証で最適値を探す（第13回で扱う）。`alpha` が大きいほど正則化が強くなり，係数は0に近づく。

### 特徴量が多いほど正則化が効く

問題1で多項式展開によって特徴量を増やしたため，正則化の効果が観察しやすくなっている。特に **Lasso（L1）は不要な特徴量の係数を完全に0にする**ため，`alpha` を大きくするにつれて「使われる特徴量の数（非ゼロ係数の数）」が減っていく。これが Lasso による**特徴量選択**だ。

> **発展**: `model.coef_` を直接見ると，どの特徴量がゼロになったか（捨てられたか）を確認できる。

### 課題

下のコードセルは実験ループの骨組みを用意してありますが、**核心（モデルの学習）はあなたが書きます**（`# ★あなたが書く★`）。問題1で作った過学習しやすいデータ（`X_train_s` / `X_test_s`）に対して、正則化強度 `alpha` を変えながら Ridge と Lasso を学習し、**訓練 R²・テスト R²・非ゼロ係数の数**を出力します。

`alphas` の値を **5通り以上**（小さい値〜大きい値）試し、しかも **Ridge と Lasso の両方**（＝2軸）について、結果を **✍️ 解答用コードセルの実験ログ**に記入してください。特に **Lasso では alpha を大きくすると非ゼロ係数の数が減っていく**様子（＝特徴量選択）に注目してください。

そのうえで考察してください：

> **考察1**: alpha を大きくすると、テスト R² はどう動きましたか？ 小さすぎ・大きすぎのどちらでもテスト R² が下がる「ちょうど良い alpha」がありましたか？
>
> **考察2（L1 と L2 の違い）**: Ridge（L2）の非ゼロ係数の数はほとんど変わらないのに、**Lasso（L1）は alpha を大きくすると係数がゼロになっていく**のはなぜですか？「Lasso が特徴量選択になる」とはどういう意味か、解答用コードセルに書いてください。（ヒント：L1 は `Σ|係数|`、L2 は `Σ(係数²)` にペナルティをかける。0付近での効き方の違いを考える）


In [ ]:
# === 完成済みコード：alpha を変えて Ridge / Lasso の係数とテスト R² を観察する ===
# 参考：正則化なしの LinearRegression（問題1の lr）と比べてください
print(f"[参考] LinearRegression  test_R2={r2_score(y_test, lr.predict(X_test_s)):.4f}  "
      f"非ゼロ係数={np.sum(np.abs(lr.coef_) > 1e-8)}/{len(lr.coef_)}\n")

# === ★ここを変えて実験する★：alpha の候補（正則化の強さ）===
alphas = [0.01, 0.1, 1, 10, 100]

header = f"{'model':6s} {'alpha':>7s} {'train_R2':>9s} {'test_R2':>9s} {'非ゼロ係数':>12s}"
print(header)
print("-" * len(header))
for alpha in alphas:
    for name, model in [("Ridge", Ridge(alpha=alpha)),
                        ("Lasso", Lasso(alpha=alpha, max_iter=10000))]:
        # ★あなたが書く★：model を訓練データ（X_train_s, y_train）で学習する（1行）
        #   ヒント: model.fit(説明変数, 目的変数)
        ___
        tr = r2_score(y_train, model.predict(X_train_s))
        te = r2_score(y_test, model.predict(X_test_s))
        n_nonzero = np.sum(np.abs(model.coef_) > 1e-8)
        print(f"{name:6s} {alpha:7.2f} {tr:9.4f} {te:9.4f} {n_nonzero:8d}/{len(model.coef_)}")


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 実験ログ（5通り以上・Ridge と Lasso の両方を記入）
experiment_log = pd.DataFrame([
    {'row': 1, 'model': 'Lasso', 'alpha': 0.01, 'train_r2': None, 'test_r2': None, 'n_nonzero': None},
    {'row': 2, 'model': 'Lasso', 'alpha': 1, 'train_r2': None, 'test_r2': None, 'n_nonzero': None},
    {'row': 3, 'model': 'Lasso', 'alpha': 100, 'train_r2': None, 'test_r2': None, 'n_nonzero': None},
    {'row': 4, 'model': 'Ridge', 'alpha': 0.01, 'train_r2': None, 'test_r2': None, 'n_nonzero': None},
    {'row': 5, 'model': 'Ridge', 'alpha': 100, 'train_r2': None, 'test_r2': None, 'n_nonzero': None},
    {'row': 6, 'model': None, 'alpha': None, 'train_r2': None, 'test_r2': None, 'n_nonzero': None},
])

# (2-b) Lasso で alpha を大きくしたとき、非ゼロ係数の数はどう変化したか
answer_2_b = """
"""

# (2-c) 考察1：alpha とテスト R² の関係（ちょうど良い alpha はあったか）
reflection1 = """
"""

# (2-d) 考察2：L1 と L2 の違い（なぜ Lasso は係数がゼロになる＝特徴量選択になるのか）
reflection2 = """
"""



## 問題3　ロジスティック回帰（正則化つき分類）を説明する　【説明】
***

### 回帰から分類へ

「点数を予測する（回帰）」から「合格/不合格を予測する（分類）」に問題を変換する。機械学習の実務では，最終的な判断が「Yes/No」「合格/不合格」「異常/正常」であることが多く，この変換はよく行われる。

### ロジスティック回帰とは

名前に「回帰」が入っているが，実際は**分類モデル**だ。内部では：

1. 線形結合を計算: z = w₁x₁ + w₂x₂ + ... + b
2. シグモイド関数で確率に変換: P(y=1) = 1 / (1 + e^(-z))
3. P > 0.5 なら 1（合格），P ≤ 0.5 なら 0（不合格）と予測

シグモイド関数は出力を必ず [0, 1] の範囲に収めるため，確率として解釈できる。

### 説明変数の選択について

今回使う変数の意味：
- `G1`, `G2`: 1〜2学期の成績（最も予測力が高いはず）
- `studytime`: 週の勉強時間（1=2時間未満 〜 4=10時間以上）
- `failures`: 過去の留年回数

> **考えてみよう**: なぜ `G1`, `G2` だけでなく `studytime` や `failures` も加えるのか？単純に成績だけでなく「学習習慣」や「過去の失敗」が最終成績の予測に寄与するかを確かめたいからだ。

### 課題

ロジスティック回帰も実は**正則化つきのモデル**です（`LogisticRegression` はデフォルトで L2 正則化がかかっています）。ここでは「合格/不合格」を予測する分類モデルの**構築コードの中身を理解**します。

下のコードセルは **完成形**です。**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたらセルを実行し、エラーなく正解率が出ることを確認してください。

説明を書くときは、次の問いを意識してください：

- `penalty="l2"` は何にペナルティをかけているか？（問題2の Ridge と同じ考え方）
- `C` は正則化の強さと**どういう関係**か？（`alpha` と逆向きだったことを思い出す）
- `solver`（最適化アルゴリズム）と `max_iter` は何を決めているか？なぜ `max_iter` を大きくするのか？

> **設計判断（一言で）**: もし `C` を `1.0` から `0.001`（とても小さい値）に変えたら、係数とモデルの複雑さはどうなると思いますか？ 解答用コードセルに1〜2文で書いてください。

In [ ]:
# 各行の「# 説明:」に自分の言葉で意味を書いてください（コードは変更しない）。
# （説明は AI に書かせず、自分で書くこと）

y_clf = (df["G3"] >= 10).astype(int)                  # 説明:（G3>=10 を何に変換している？）
clf_cols = ["G1", "G2", "studytime", "failures"]
Xc = df[clf_cols].values

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, y_clf, test_size=0.2, random_state=0, stratify=y_clf  # 説明:（stratify は何のため？）
)

clf_scaler = StandardScaler()
Xc_train_s = clf_scaler.fit_transform(Xc_train)       # 説明:（訓練データで fit する理由）
Xc_test_s = clf_scaler.transform(Xc_test)             # 説明:（test は transform だけにする理由）

logreg = LogisticRegression(
    penalty="l2",     # 説明:（何にペナルティをかける？）
    C=1.0,            # 説明:（C と正則化の強さの関係）
    solver="lbfgs",   # 説明:（solver は何を決める？）
    max_iter=1000,    # 説明:（なぜ大きくするのか）
)
logreg.fit(Xc_train_s, yc_train)                      # 説明:
acc = accuracy_score(yc_test, logreg.predict(Xc_test_s))  # 説明:
print("テスト正解率 =", round(acc, 4))


In [ ]:
# === ✍️ 問題3 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (3-a) `penalty="l2"` は何にペナルティをかけているか
answer_3_a = """
"""

# (3-b) `C` は正則化の強さとどういう関係か（`alpha` と逆向きだったことを踏まえて）
answer_3_b = """
"""

# (3-c) `solver` と `max_iter` は何を決めているか
answer_3_c = """
"""

# (3-d) 設計判断：`C` を `1.0` から `0.001` に変えると、係数とモデルの複雑さはどうなるか（1〜2文）
answer_3_d = """
"""



## 問題4　正則化強度 C の比較実験　【実験】
***

### ハイパーパラメータ探索の考え方

C の値（正則化強度）は事前に「最適値」がわからない。そのため，複数の値を試して**テストデータでの性能を比較**する。これをハイパーパラメータ探索（Grid Search）という（第13回で自動化する方法を学ぶ）。

### なぜ対数スケールで比較するか

C = [0.01, 0.1, 1, 10, 100] は1000倍の範囲を探索している。線形スケールでは 0.01, 0.1, 1 が密集して見えてしまう。対数スケールにすることで各値を等間隔に表示でき，変化の傾向が見やすくなる。

### グラフから何を読み取るか

このグラフで典型的に見られるパターン：
- **C が小さすぎる（左端）**: 正則化が強すぎて underfitting（過少適合）→ 訓練もテストも低精度
- **C が大きすぎる（右端）**: 正則化が弱すぎて overfitting（過学習）→ 訓練は高精度だがテストは低下
- **最適な C**: テスト精度が最大になるあたり

> **卒業研究での応用**: このようなグラフを描いて最適なハイパーパラメータを決定するプロセスは，あらゆる機械学習研究で必要になる。論文にはこのような「ハイパーパラメータ感度分析」が含まれることが多い。

### 課題

下のコードセルはグラフ描画まで用意してありますが、**核心（各 C でモデルを作って学習する処理）はあなたが書きます**（`# ★あなたが書く★`）。問題3で作った分類データに対して、`C` を変えながら**訓練正解率とテスト正解率の両方**を計算し、折れ線グラフ（横軸 C は対数スケール）を描いて、テスト正解率が最大の C を出力します。

`Cs` の値を **5通り以上**試し、結果を **✍️ 解答用コードセルの実験ログ**に記入してください。さらに 2つ目の軸として、**`penalty="l1"`（`solver="liblinear"`）でも**少なくとも1〜2点試して L2 と比べてみましょう。

そのうえで考察してください：

> **考察3**: C を小さくする（正則化を強める）と訓練正解率はどう動きましたか？ C を大きくする（正則化を弱める）と、訓練とテストの差はどうなりましたか？ グラフのどのあたりが「ちょうど良い C」でしたか？
>
> **考察4**: 問題2の Ridge/Lasso の `alpha` と、この問題の `C` は**逆向き**の関係でした。「`alpha` を大きくする」と「`C` を大きくする」は、正則化の強さの観点でそれぞれどちらに向かうのか、解答用コードセルに書いてください。

In [ ]:
# === 完成済みコード：C を変えて訓練・テスト正解率の変化を観察する ===
# === ★ここを変えて実験する★：C の候補（大きいほど正則化が弱い）===
Cs = [0.01, 0.1, 1, 10, 100]

train_accs, test_accs = [], []
for C in Cs:
    # ★あなたが書く★：正則化強度 C のロジスティック回帰 m を作り、学習する（2行）
    #   ヒント: LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=1000) を作り .fit
    m = ___
    ___
    train_accs.append(accuracy_score(yc_train, m.predict(Xc_train_s)))
    test_accs.append(accuracy_score(yc_test, m.predict(Xc_test_s)))
    print(f"C={C:7.2f}  train_acc={train_accs[-1]:.4f}  test_acc={test_accs[-1]:.4f}")

plt.plot(Cs, train_accs, "o-", label="train")
plt.plot(Cs, test_accs, "s-", label="test")
plt.xscale("log")
plt.xlabel("C（対数スケール，大きいほど正則化が弱い）")
plt.ylabel("正解率")
plt.title("正則化強度 C と正解率")
plt.legend()
plt.show()

best_C = Cs[int(np.argmax(test_accs))]
print("テスト正解率が最大の C =", best_C)


In [ ]:
# === ✍️ 問題4 解答（採点対象）===
import pandas as pd


# (4-a) 実験ログ（5通り以上。penalty="l1" も1〜2点）
experiment_log = pd.DataFrame([
    {'row': 1, 'penalty': 'l2', 'C': 0.01, 'train_acc': None, 'test_acc': None},
    {'row': 2, 'penalty': 'l2', 'C': 0.1, 'train_acc': None, 'test_acc': None},
    {'row': 3, 'penalty': 'l2', 'C': 1, 'train_acc': None, 'test_acc': None},
    {'row': 4, 'penalty': 'l2', 'C': 100, 'train_acc': None, 'test_acc': None},
    {'row': 5, 'penalty': 'l1', 'C': 1, 'train_acc': None, 'test_acc': None},
    {'row': 6, 'penalty': None, 'C': None, 'train_acc': None, 'test_acc': None},
])

# (4-b) テスト正解率が最大だった C
answer_4_b = """
"""

# (4-c) 考察3：C を変えると訓練／テスト正解率はどう動いたか・ちょうど良い C
reflection3 = """
"""

# (4-d) 考察4：`alpha` を大きくする と `C` を大きくする は、正則化の強さでどちら向きか
reflection4 = """
"""

